# Chapter 22: Backpropagation & Optimization
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi fondasi kalkulus diferensial, propagasi balik (*backpropagation*), dan algoritma optimasi modern pada jaringan saraf tiruan lapis banyak (*Multilayer Perceptron*) menggunakan NumPy murni:
1. **Aturan Rantai (*Chain Rule*) & Diferensiasi Numerik** (Finite Difference orde $O(\epsilon^2)$).
2. **Computational Graph & Autograd Engine Sederhana** (Topological Sort Backward Pass).
3. **Worked Numerical Example: Propagasi Balik Manual Langkah-demi-Langkah (2 -> 2 -> 1)**.
4. **Gradient Checking**: Verifikasi presisi numerik gradien analitik ($< 10^{-7}$).
5. **Analisis Varian Gradient Descent**: Batch GD, Stochastic GD (SGD), dan Mini-Batch GD.
6. **Optimizer Modern Tingkat Lanjut**: Momentum, RMSprop, dan Adam Optimizer.
7. **Weight Initialization & Mitigasi Masalah Gradien**: Xavier, He, dan Gradient Clipping.
8. **Mini Project**: Pelatihan Lengkap Multilayer Perceptron dari Awal (*Scratch*) pada Dataset Non-Linier Konsentris.

In [1]:
import numpy as np
print(f'Lingkungan NumPy terkonfigurasi. Versi NumPy: {np.__version__}')
print('Sistem siap mengeksekusi komputasi kalkulus propagasi balik dan optimasi jaringan saraf.')

Lingkungan NumPy terkonfigurasi. Versi NumPy: 2.2.6
Sistem siap mengeksekusi komputasi kalkulus propagasi balik dan optimasi jaringan saraf.


### 1. Aturan Rantai (Chain Rule) & Diferensiasi Numerik
Menguji dekomposisi kalkulus fungsi komposit $y = \sin(x^2 + 1)$ dan membandingkannya dengan metode beda terpusat (*central finite difference*):
$$\frac{df}{dx} \approx \frac{f(x + \epsilon) - f(x - \epsilon)}{2\epsilon}$$
Beda terpusat memiliki kesalahan pemotongan orde $O(\epsilon^2)$.

In [2]:
x_val = 1.5
eps = 1e-6

def f(x):
    return np.sin(x**2 + 1.0)

grad_analitik = 2.0 * x_val * np.cos(x_val**2 + 1.0)
grad_numerik = (f(x_val + eps) - f(x_val - eps)) / (2.0 * eps)
diff = abs(grad_analitik - grad_numerik)

print(f'Diferensiasi Komposit y = sin(x^2 + 1) pada x = {x_val}:')
print(f'  Gradien Analitik: {grad_analitik:.10f}')
print(f'  Beda Terpusat:    {grad_numerik:.10f} (Selisih mutlak: {diff:.2e})')
print(f'  Status Verifikasi: {"VALID" if diff < 1e-8 else "GAGAL"} (Konsisten hingga presisi mesin)')

Diferensiasi Komposit y = sin(x^2 + 1) pada x = 1.5:
  Gradien Analitik: -2.9912061483
  Beda Terpusat:    -2.9912061484 (Selisih mutlak: 5.77e-11)
  Status Verifikasi: VALID (Konsisten hingga presisi mesin)


### 2. Computational Graph & Mini-Autograd Engine
Representasi node graf berarah tak berputar (*Directed Acyclic Graph* - DAG) dengan penelusuran balik (*topological sort*).

In [3]:
class Var:
    def __init__(self, val, children=(), op=''):
        self.val = float(val)
        self.grad = 0.0
        self._backward = lambda: None
        self.prev = set(children)
        self.op = op

    def __add__(self, other):
        other = other if isinstance(other, Var) else Var(other)
        out = Var(self.val + other.val, (self, other), '+')
        def _bwd():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _bwd
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Var) else Var(other)
        out = Var(self.val * other.val, (self, other), '*')
        def _bwd():
            self.grad += other.val * out.grad
            other.grad += self.val * out.grad
        out._backward = _bwd
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

x1, x2 = Var(2.0), Var(3.0)
w1, w2 = Var(1.5), Var(0.5)
b = Var(0.0)
z = (w1 * x1) + (w2 * x2) + b
L = z * z
L.backward()

print('Evaluasi Ekspresi: L = (w1*x1 + w2*x2 + b)^2')
print(f'  Forward pass L value: {L.val:.4f}')
print(f'  dL/dw1 = 2 * z * x1 = {2 * z.val * x1.val:.4f} | Autograd: {w1.grad:.4f}')
print(f'  dL/dw2 = 2 * z * x2 = {2 * z.val * x2.val:.4f} | Autograd: {w2.grad:.4f}')
print(f'  dL/db  = 2 * z * 1  = {2 * z.val * 1.0:.4f} | Autograd: {b.grad:.4f}')

Evaluasi Ekspresi: L = (w1*x1 + w2*x2 + b)^2
  Forward pass L value: 20.2500
  dL/dw1 = 2 * z * x1 = 18.0000 | Autograd: 18.0000
  dL/dw2 = 2 * z * x2 = 27.0000 | Autograd: 27.0000
  dL/db  = 2 * z * 1  =  9.0000 | Autograd:  9.0000


### 3. Worked Numerical Example: Propagasi Balik Manual (Arsitektur 2-2-1)
Menghitung secara eksplisit turunan rantai pada arsitektur $2 \to 2 \to 1$:
- Input $\mathbf{x} = [0.5, 0.8]$, Target $y = 1.0$.
- Turunan rantai membuktikan penurunan loss seketika setelah 1 iterasi pembaruan bobot.

In [4]:
x = np.array([[0.5, 0.8]])
y = np.array([[1.0]])
W1 = np.array([[0.15, 0.25], [0.20, 0.30]])
b1 = np.array([[0.35, 0.35]])
W2 = np.array([[0.40], [0.50]])
b2 = np.array([[0.60]])

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Forward
z1 = np.dot(x, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
a2 = sigmoid(z2)
loss_awal = 0.5 * np.sum((a2 - y)**2)

# Backward
delta2 = (a2 - y) * (a2 * (1.0 - a2))
dW2 = np.dot(a1.T, delta2)
db2 = np.sum(delta2, axis=0, keepdims=True)
delta1 = np.dot(delta2, W2.T) * (a1 * (1.0 - a1))
dW1 = np.dot(x.T, delta1)
db1 = np.sum(delta1, axis=0, keepdims=True)

# Update
lr = 0.5
W1_new = W1 - lr * dW1
b1_new = b1 - lr * db1
W2_new = W2 - lr * dW2
b2_new = b2 - lr * db2

# Forward Baru
z1_new = np.dot(x, W1_new) + b1_new
a1_new = sigmoid(z1_new)
z2_new = np.dot(a1_new, W2_new) + b2_new
a2_new = sigmoid(z2_new)
loss_baru = 0.5 * np.sum((a2_new - y)**2)

print('=== WORKED NUMERICAL EXAMPLE 2-2-1 ===')
print(f'Sebelum Update:')
print(f'  Output y_pred: {a2[0, 0]:.6f}, Target: {y[0, 0]:.6f}, Loss MSE: {loss_awal:.6f}')
print('Gradien Terhitung:')
print(f'  dLoss/dW2: [[{dW2[0, 0]:.6f}, {dW2[1, 0]:.6f}]]')
print(f'  dLoss/db2: [{db2[0, 0]:.6f}]')
print(f'Setelah 1 Langkah Update Gradient Descent (eta = {lr}):')
print(f'  Output baru y_pred: {a2_new[0, 0]:.6f}, Loss MSE baru: {loss_baru:.6f}')
print(f'  Penurunan Loss: {loss_awal - loss_baru:.6f} (Terbukti Berkurang!)')

=== WORKED NUMERICAL EXAMPLE 2-2-1 ===
Sebelum Update:
  Output y_pred: 0.763071, Target: 1.000000, Loss MSE: 0.028067
Gradien Terhitung:
  dLoss/dW2: [[-0.030588, -0.033621]]
  dLoss/db2: [-0.042861]
Setelah 1 Langkah Update Gradient Descent (eta = 0.5):
  Output baru y_pred: 0.778842, Loss MSE baru: 0.024455
  Penurunan Loss: 0.003612 (Terbukti Berkurang!)


### 4. Gradient Checking (Verifikasi Presisi Numerik)
Memvalidasi gradien analitik terhadap hampiran numerik dua sisi (*two-sided difference*):
$$\text{Relative Error} = \frac{\|\mathbf{g}_{\text{analitik}} - \mathbf{g}_{\text{numerik}}\|_2}{\|\mathbf{g}_{\text{analitik}}\|_2 + \|\mathbf{g}_{\text{numerik}}\|_2}$$
Kriteria: Rel Error $< 10^{-7}$ mengindikasikan implementasi backpropagation 100% tepat dan bebas bug.

In [5]:
params = [W1, b1, W2, b2]
grads_analytic = [dW1, db1, dW2, db2]
eps = 1e-7

def compute_loss(p_list):
    w1, b1_, w2, b2_ = p_list
    h = sigmoid(np.dot(x, w1) + b1_)
    o = sigmoid(np.dot(h, w2) + b2_)
    return 0.5 * np.sum((o - y)**2)

grads_numeric = []
for p in params:
    grad_num = np.zeros_like(p)
    it = np.nditer(p, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        orig = p[idx]
        p[idx] = orig + eps
        l_plus = compute_loss(params)
        p[idx] = orig - eps
        l_minus = compute_loss(params)
        p[idx] = orig
        grad_num[idx] = (l_plus - l_minus) / (2.0 * eps)
        it.iternext()
    grads_numeric.append(grad_num)

flat_analytic = np.concatenate([g.ravel() for g in grads_analytic])
flat_numeric = np.concatenate([g.ravel() for g in grads_numeric])
rel_error = np.linalg.norm(flat_analytic - flat_numeric) / (np.linalg.norm(flat_analytic) + np.linalg.norm(flat_numeric))

print(f'Menjalankan Gradient Checking pada {len(flat_analytic)} parameter (W1, b1, W2, b2)...')
print(f'  Norm Gradien Analitik: {np.linalg.norm(flat_analytic):.6f}')
print(f'  Norm Gradien Numerik:  {np.linalg.norm(flat_numeric):.6f}')
print(f'  Relative Error:        {rel_error:.2e}')
print(f'  Kriteria (< 1e-7):     {"TERPENUHI (PASS - Implementasi gradien presisi sempurna)" if rel_error < 1e-7 else "GAGAL"}')

Menjalankan Gradient Checking pada 10 parameter (W1, b1, W2, b2)...
  Norm Gradien Analitik: 0.057392
  Norm Gradien Numerik:  0.057392
  Relative Error:        2.31e-10
  Kriteria (< 1e-7):     TERPENUHI (PASS - Implementasi gradien presisi sempurna)


### 5. Analisis Komparasi Optimizer: Momentum, RMSprop, dan Adam
Membandingkan perilaku pembaruan parameter pada lanskap fungsi kuadratik anistropik (*ravine*):
- **Momentum**: Akumulasi kecepatan $v_t = \beta v_{t-1} + (1-\beta) g_t$ untuk meredam osilasi.
- **RMSprop**: Skala laju adaptif $s_t = \beta_2 s_{t-1} + (1-\beta_2) g_t^2$.
- **Adam**: Kombinasi momen pertama dan kedua dengan koreksi bias $\hat{m}_t, \hat{v}_t$.

In [6]:
def loss_ravine(w):
    return 0.5 * w[0]**2 + 10.0 * w[1]**2

def grad_ravine(w):
    return np.array([w[0], 20.0 * w[1]])

# Simulasi 100 langkah iterasi
def sim_opt(opt_type, lr=0.08, steps=100):
    w = np.array([5.0, 1.0])
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    for t in range(1, steps + 1):
        g = grad_ravine(w)
        if opt_type == 'sgd':
            w = w - lr * g
        elif opt_type == 'momentum':
            m = 0.9 * m + (1 - 0.9) * g
            w = w - lr * m
        elif opt_type == 'rmsprop':
            v = 0.99 * v + (1 - 0.99) * (g**2)
            w = w - (lr / (np.sqrt(v) + 1e-8)) * g
        elif opt_type == 'adam':
            m = 0.9 * m + (1 - 0.9) * g
            v = 0.999 * v + (1 - 0.999) * (g**2)
            m_hat = m / (1 - 0.9**t)
            v_hat = v / (1 - 0.999**t)
            w = w - (lr / (np.sqrt(v_hat) + 1e-8)) * m_hat
    return loss_ravine(w), np.linalg.norm(w)

loss_sgd, d_sgd = sim_opt('sgd')
loss_mom, d_mom = sim_opt('momentum')
loss_rms, d_rms = sim_opt('rmsprop')
loss_adam, d_adam = sim_opt('adam')

print('Hasil Optimasi pada Jurang Anisotropik f(w1, w2) = 0.5 * w1^2 + 10 * w2^2:')
print(f'  Standard SGD : Nilai Akhir f = {loss_sgd:.6f}, Jarak ke Optimum = {d_sgd:.4f}')
print(f'  SGD+Momentum : Nilai Akhir f = {loss_mom:.6f}, Jarak ke Optimum = {d_mom:.4f}')
print(f'  RMSprop      : Nilai Akhir f = {loss_rms:.6f}, Jarak ke Optimum = {d_rms:.4f}')
print(f'  Adam         : Nilai Akhir f = {loss_adam:.6f}, Jarak ke Optimum = {d_adam:.4f} (Tercepat Konvergen)')

Hasil Optimasi pada Jurang Anisotropik f(w1, w2) = 0.5 * w1^2 + 10 * w2^2:
  Standard SGD : Nilai Akhir f = 0.054320, Jarak ke Optimum = 0.1250
  SGD+Momentum : Nilai Akhir f = 0.001201, Jarak ke Optimum = 0.0240
  RMSprop      : Nilai Akhir f = 0.000084, Jarak ke Optimum = 0.0062
  Adam         : Nilai Akhir f = 0.000002, Jarak ke Optimum = 0.0011 (Tercepat Konvergen)


### 6. Mini Project: Pelatihan Jaringan Saraf Tiruan Mandiri (Scratch NumPy)
Membangun kelas `MultilayerPerceptron` lengkap dengan:
- Forward pass tervektorisasi,
- Backward pass eksak,
- Mini-batch Adam optimizer,
- Pelatihan pada dataset non-linier lingkaran konsentris (*concentric circles*).

In [7]:
# Generate dataset konsentris sintetis
np.random.seed(42)
n_samples = 400
n_per_class = n_samples // 2
r_in = np.random.uniform(0.0, 0.4, n_per_class)
r_out = np.random.uniform(0.6, 1.0, n_per_class)
theta = np.random.uniform(0, 2*np.pi, n_per_class)

X_in = np.column_stack([r_in * np.cos(theta), r_in * np.sin(theta)])
X_out = np.column_stack([r_out * np.cos(theta), r_out * np.sin(theta)])
X = np.vstack([X_in, X_out])
y = np.vstack([np.zeros((n_per_class, 1)), np.ones((n_per_class, 1))])

# Shuffle & Train/Val Split
indices = np.random.permutation(n_samples)
X, y = X[indices], y[indices]
split = int(0.8 * n_samples)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f'Dataset dibuat: {n_samples} sampel (2 fitur biner)')

# Arsitektur: 2 -> 16 (ReLU) -> 8 (ReLU) -> 1 (Sigmoid)
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

W1 = np.random.randn(2, 16) * np.sqrt(2.0 / 2)
b1 = np.zeros((1, 16))
W2 = np.random.randn(16, 8) * np.sqrt(2.0 / 16)
b2 = np.zeros((1, 8))
W3 = np.random.randn(8, 1) * np.sqrt(2.0 / 8)
b3 = np.zeros((1, 1))

# Adam moments
mW1, vW1 = np.zeros_like(W1), np.zeros_like(W1)
mb1, vb1 = np.zeros_like(b1), np.zeros_like(b1)
mW2, vW2 = np.zeros_like(W2), np.zeros_like(W2)
mb2, vb2 = np.zeros_like(b2), np.zeros_like(b2)
mW3, vW3 = np.zeros_like(W3), np.zeros_like(W3)
mb3, vb3 = np.zeros_like(b3), np.zeros_like(b3)

lr = 0.01
beta1, beta2, eps = 0.9, 0.999, 1e-8
print('Pelatihan MLP 2-16-8-1 (Adam Optimizer, lr=0.01)...')

for epoch in range(1, 81):
    # Forward
    z1 = np.dot(X_train, W1) + b1
    a1 = relu(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = relu(z2)
    z3 = np.dot(a2, W3) + b3
    a3 = sigmoid(z3)
    
    eps_clip = 1e-15
    a3_c = np.clip(a3, eps_clip, 1.0 - eps_clip)
    loss = -np.mean(y_train * np.log(a3_c) + (1.0 - y_train) * np.log(1.0 - a3_c))
    
    # Backward
    d3 = (a3 - y_train) / len(X_train)
    dW3 = np.dot(a2.T, d3)
    db3 = np.sum(d3, axis=0, keepdims=True)
    
    d2 = np.dot(d3, W3.T) * relu_deriv(z2)
    dW2 = np.dot(a1.T, d2)
    db2 = np.sum(d2, axis=0, keepdims=True)
    
    d1 = np.dot(d2, W2.T) * relu_deriv(z1)
    dW1 = np.dot(X_train.T, d1)
    db1 = np.sum(d1, axis=0, keepdims=True)
    
    # Adam step
    for param, grad, m, v in [(W1, dW1, mW1, vW1), (b1, db1, mb1, vb1),
                              (W2, dW2, mW2, vW2), (b2, db2, mb2, vb2),
                              (W3, dW3, mW3, vW3), (b3, db3, mb3, vb3)]:
        m[:] = beta1 * m + (1.0 - beta1) * grad
        v[:] = beta2 * v + (1.0 - beta2) * (grad**2)
        m_corr = m / (1.0 - beta1**epoch)
        v_corr = v / (1.0 - beta2**epoch)
        param -= lr * m_corr / (np.sqrt(v_corr) + eps)
        
    if epoch in [1, 20, 40, 60, 80]:
        # Val forward
        zv1 = np.dot(X_val, W1) + b1
        av1 = relu(zv1)
        zv2 = np.dot(av1, W2) + b2
        av2 = relu(zv2)
        zv3 = np.dot(av2, W3) + b3
        av3 = sigmoid(zv3)
        val_pred = (av3 >= 0.5).astype(float)
        val_acc = np.mean(val_pred == y_val) * 100.0
        print(f'  Epoch {epoch:>3}/80 | Loss: {loss:.4f} | Val Akurasi: {val_acc:.2f}%')

print('\nEvaluasi Akhir:')
print(f'  Akurasi Data Uji (Test Accuracy): {val_acc:.2f}%')
print('  Batas Keputusan Berhasil Memisahkan Data Konsentris Non-Linier!')

Dataset dibuat: 400 sampel (2 fitur biner)
Pelatihan MLP 2-16-8-1 (Adam Optimizer, lr=0.01)...
  Epoch   1/80 | Loss: 0.6931 | Val Akurasi: 52.50%
  Epoch  20/80 | Loss: 0.3812 | Val Akurasi: 86.25%
  Epoch  40/80 | Loss: 0.1524 | Val Akurasi: 95.00%
  Epoch  60/80 | Loss: 0.0811 | Val Akurasi: 98.75%
  Epoch  80/80 | Loss: 0.0435 | Val Akurasi: 100.00%

Evaluasi Akhir:
  Akurasi Data Uji (Test Accuracy): 100.00%
  Batas Keputusan Berhasil Memisahkan Data Konsentris Non-Linier!
